Pipeline в библиотеке Transformers от Hugging Face — это высокоуровневый интерфейс, который упрощает использование предобученных моделей для разных задач NLP. Pipeline инкапсулирует все шаги в один простой вызов функции, а значит, не нужно писать сложный код для каждого шага. 

Pipeline автоматически:
- загружает подходящий токенизатор и модель,
- токенизирует входной текст,
- пропускает токены через модель,
- постобрабатывает выходные данные в понятном формате.

In [1]:
from transformers import pipeline

# Создание pipeline с автоматическим выбором модели
classifier = pipeline("sentiment-analysis")

# Использование pipeline
result = classifier("I love this product!")
print(result)  # [{'label': 'POSITIVE', 'score': 0.9998}]

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11565.89it/s]


[{'label': 'POSITIVE', 'score': 0.9998855590820312}]


In [2]:
from transformers import pipeline

# Создание pipeline с конкретной моделью
classifier = pipeline("sentiment-analysis", 
                     model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

# Обработка текста на разных языках
texts = [
    "I love this!",           # английский
    "Мне это нравится!",      # русский  
    "¡Me encanta esto!"       # испанский
]

for text in texts:
    result = classifier(text)
    print(f"{text} -> {result[0]['label']} ({result[0]['score']:.3f})")

# I love this! -> positive (0.938)
# Мне это нравится! -> positive (0.914)
# ¡Me encanta esto! -> positive (0.941)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35669.77it/s]


I love this! -> positive (0.938)
Мне это нравится! -> positive (0.914)
¡Me encanta esto! -> positive (0.941)


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Загружаем компоненты отдельно
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-xlm-roberta-base-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-xlm-roberta-base-sentiment")

# Создаём pipeline из компонентов
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

for text in texts:
    result = classifier(text)
    print(f"{text} -> {result[0]['label']} ({result[0]['score']:.3f})")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35771.18it/s]


I love this! -> positive (0.938)
Мне это нравится! -> positive (0.914)
¡Me encanta esto! -> positive (0.941)


## Задание 1
Пронаблюдайте, как качественно XLM-R работает на задаче классификации тональности текстов на разных языках. 

Допишите код, используйте предобученную модель XLM-RoBERTa для определения эмоциональной окраски отзывов на английском, русском и испанском языках без дополнительного обучения.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

# Загружаем токенизатор и модель XLM-R для классификации тональности
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Создаём pipeline для классификации
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Тестовые тексты на разных языках
texts = [
    "This movie is absolutely fantastic! I loved every moment of it.",  # английский
    "Этот фильм просто ужасен, потратил время зря.",  # русский  
    "¡Me encanta este producto! Es increíble y muy útil.",  # испанский
    "I'm not sure about this book, it's okay I guess.",  # английский
    "Сервис отличный, всем рекомендую!",  # русский
    "No me gusta nada, muy decepcionante."  # испанский
]

# Классифицируем тональность для каждого текста
for i, text in enumerate(texts):
    result = classifier(text)
    print(f"Текст {i+1}: {text}")
    print(f"Тональность: {result[0]['label']} (уверенность: {result[0]['score']:.3f})")
    print("-" * 50)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 37055.74it/s]


Текст 1: This movie is absolutely fantastic! I loved every moment of it.
Тональность: positive (уверенность: 0.949)
--------------------------------------------------
Текст 2: Этот фильм просто ужасен, потратил время зря.
Тональность: negative (уверенность: 0.935)
--------------------------------------------------
Текст 3: ¡Me encanta este producto! Es increíble y muy útil.
Тональность: positive (уверенность: 0.950)
--------------------------------------------------
Текст 4: I'm not sure about this book, it's okay I guess.
Тональность: neutral (уверенность: 0.639)
--------------------------------------------------
Текст 5: Сервис отличный, всем рекомендую!
Тональность: positive (уверенность: 0.907)
--------------------------------------------------
Текст 6: No me gusta nada, muy decepcionante.
Тональность: negative (уверенность: 0.952)
--------------------------------------------------


## Задание 2
Пронаблюдайте силу DeBERTa в задаче Natural Language Inference (NLI) — распознавании логических отношений между парой предложений. Мы подобрали примеры, которые помогут увидеть, как модель справляется с ловушками: лексической неоднозначностью, контекстно-зависимыми местоимениями и длинными зависимостями. 

В модели уже встроены знания о семантике и длинных зависимостях (предобучение + дообучение на датасете MNLI/Fever/ANLI), поэтому предлагаем пронаблюдать работу модели на английских парах (premise + hypothesis). Для удобства и наглядности мы подготовили переводы на русский язык (premise_ru / hypothesis_ru) и выведем результаты на нём.

### Что нужно сделать
1. Загрузить токенизатор и модель `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`.
2. Для каждой пары `(premise, hypothesis)` токенизировать английские версии:
   - в правильном порядке: `premise` — первым, `hypothesis` — вторым;
   - используя параметры `return_tensors="pt", padding=True, truncation=True, max_length=512`.
3. Передать тензоры в модель, получить `logits` и превратить их в вероятности с помощью `softmax`.
4. Корректно сопоставить индексы выходов модели с метками (`entailment`, `neutral` или `contradiction`). Не полагайтесь на фиксированный порядок — проверьте `model.config.id2label` и постройте русские метки в том же порядке.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Загружаем модель и токенизатор
model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Сложные пары предложений для анализа с переводами
pairs = [
    {
        "premise_en": "The bank by the river was steep and muddy.",
        "hypothesis_en": "The financial institution was near the water.",
        "premise_ru": "Берег у реки был крутым и грязным.",
        "hypothesis_ru": "Финансовое учреждение было рядом с водой."
    },
    {
        "premise_en": "The doctor advised the lawyer because she felt unwell.",
        "hypothesis_en": "The lawyer was feeling unwell.",
        "premise_ru": "Доктор дал совет адвокату, потому что она плохо себя чувствовала.",
        "hypothesis_ru": "Адвокат плохо себя чувствовал."
    },
    {
        "premise_en": "Despite initial promising results announced in the press conference, the drug failed in clinical trials because of unexpected side effects observed in elderly patients.",
        "hypothesis_en": "Side effects caused the drug to fail.",
        "premise_ru": "Несмотря на первоначальные многообещающие результаты, объявленные на пресс-конференции, препарат провалился в клинических испытаниях из-за неожиданных побочных эффектов, наблюдаемых у пожилых пациентов.",
        "hypothesis_ru": "Побочные эффекты стали причиной провала препарата."
    }
]

# Классифицируем отношения
for pair in pairs:
    # Токенизация и подготовка ввода (используем английские версии!)
    # Используйте tokenizer для преобразования текста в тензоры
    # В токенизаторе передавайте оба предложения и используйте return_tensors="pt", padding=True, truncation=True
    inputs = tokenizer(pair["premise_en"], pair["hypothesis_en"], return_tensors="pt", padding=True, truncation=True, max_length=512)
    
    # Подаём данные в модель
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Преобразуем выходы в вероятности
    probs = F.softmax(outputs.logits, dim=-1)
    probs = probs[0].cpu().numpy()
    
    # Определяем метку с максимальной вероятностью
    labels_ru = ["следствие", "нейтрально", "противоречие"]
    pred_label_ru = labels_ru[probs.argmax()]
    
    # Выводим результат полностью на русском
    print(f"Событие: {pair['premise_ru']}")
    print(f"Гипотеза: {pair['hypothesis_ru']}")
    print(f"Предсказание: {pred_label_ru} (уверенность: {probs.max():.2%})")
    print(f"Вероятности: [следствие: {probs[0]:.2%}, нейтрально: {probs[1]:.2%}, противоречие: {probs[2]:.2%}]")
    print("-" * 80)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11603.30it/s]


Событие: Берег у реки был крутым и грязным.
Гипотеза: Финансовое учреждение было рядом с водой.
Предсказание: нейтрально (уверенность: 99.85%)
Вероятности: [следствие: 0.10%, нейтрально: 99.85%, противоречие: 0.05%]
--------------------------------------------------------------------------------
Событие: Доктор дал совет адвокату, потому что она плохо себя чувствовала.
Гипотеза: Адвокат плохо себя чувствовал.
Предсказание: следствие (уверенность: 98.68%)
Вероятности: [следствие: 98.68%, нейтрально: 1.23%, противоречие: 0.10%]
--------------------------------------------------------------------------------
Событие: Несмотря на первоначальные многообещающие результаты, объявленные на пресс-конференции, препарат провалился в клинических испытаниях из-за неожиданных побочных эффектов, наблюдаемых у пожилых пациентов.
Гипотеза: Побочные эффекты стали причиной провала препарата.
Предсказание: следствие (уверенность: 98.83%)
Вероятности: [следствие: 98.83%, нейтрально: 1.04%, противоречие: 0.